# Инференс SegFormer и построение 15D векторов (Фаза 4)

Этот ноутбук берёт нарезанные тайлы (800x800) и веса обученной модели SegFormer.
На выходе получается `building_features.csv` с 15-мерным вектором признаков для каждого широкого плана фасада.

In [1]:
!pip install -q transformers

In [2]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from glob import glob
from tqdm.auto import tqdm
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем: {device}")

Используем: cuda


## 1. Пути и Настройки

In [4]:
# ПУТИ В KAGGLE
TILES_DIR = '/kaggle/input/datasets/neuromant/tiles-800x800-300-imgs/tiles_800x800_inference-300' 
MODEL_PATH = '/kaggle/input/models/neuromant/segformer-b2-final-fit-408/pytorch/default/1/final_fit_408' 

EPSILON = 0.01 # Порог для prevalence (1% пикселей)

CLASS_MAPPING = {
    0: "background",
    1: "coating_deterioration",
    2: "cracks",
    3: "masonry_degradation",
    4: "moisture_bio_damage",
    5: "vandalism",
}

CLASSES_TO_TRACK = [
    "coating_deterioration", 
    "cracks", 
    "masonry_degradation", 
    "moisture_bio_damage", 
    "vandalism"
]


## 2. Загрузка модели

In [5]:
print("Загрузка процессора и модели...")
processor = SegformerImageProcessor.from_pretrained("nvidia/mit-b2")
model = SegformerForSemanticSegmentation.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()
print("Готово!")

Загрузка процессора и модели...


preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

Готово!


## 3. Инференс по тайлам

In [6]:
tile_files = glob(os.path.join(TILES_DIR, "*.jpg")) + glob(os.path.join(TILES_DIR, "*.png"))
print(f"Найдено {len(tile_files)} тайлов для обработки.")

# Здесь копятся сырые соотношения (ratios) для каждого тайла отдельно
# Формат: { "original_image_name": [ { "Mechanical": 0.05, "Bio-chemical": 0.0, ... }, ... ] }
image_data = {}

with torch.no_grad():
    for tile_path in tqdm(tile_files, desc="Инференс тайлов"):
        # Имя файла выглядит как: img123_400_800.jpg
        # Нам нужно вытащить базовое имя "img123"
        filename = os.path.basename(tile_path)
        # Отсекаем координаты _Y_X.ext
        # Например, если файл DSC_001_800_400.jpg, то base_name будет DSC_001
        parts = os.path.splitext(filename)[0].rsplit('_', 2)
        base_name = parts[0]
        
        if base_name not in image_data:
            image_data[base_name] = []
            
        # Считываем и предсказываем
        image = cv2.imread(tile_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        inputs = processor(images=image, return_tensors="pt").to(device)
        outputs = model(**inputs)
        
        logits = outputs.logits
        upsampled_logits = torch.nn.functional.interpolate(
            logits, 
            size=image.shape[:2], 
            mode="bilinear", 
            align_corners=False
        )
        
        pred_mask = upsampled_logits.argmax(dim=1).squeeze().cpu().numpy()
        
        # Считаем ratios
        total_pixels = pred_mask.size
        unique, counts = np.unique(pred_mask, return_counts=True)
        mask_counts = dict(zip(unique, counts))
        
        tile_ratios = {}
        for class_idx, class_name in CLASS_MAPPING.items():
            if class_name == "background": continue
            
            pixels = mask_counts.get(class_idx, 0)
            tile_ratios[class_name] = pixels / total_pixels
            
        image_data[base_name].append(tile_ratios)

Найдено 6693 тайлов для обработки.


Инференс тайлов:   0%|          | 0/6693 [00:00<?, ?it/s]

## 4. Агрегация признаков и построение 15D векторов

Считаем: mean, max, prevalence.

In [7]:
results = []

for base_name, tiles in tqdm(image_data.items(), desc="Агрегация 15D векторов"):
    num_tiles = len(tiles)
    vector_dict = {"building_name": base_name, "num_tiles": num_tiles}
    
    # Универсально собираем все значения ratio по каждому классу в списки
    class_ratios = {cls_name: [t.get(cls_name, 0.0) for t in tiles] for cls_name in CLASSES_TO_TRACK}

    
    # Считаем агрегации
    for x in CLASSES_TO_TRACK:
        ratios = class_ratios[x]
        
        mean_val = np.mean(ratios)
        max_val = np.max(ratios)
        prevalence = sum(1 for r in ratios if r > EPSILON) / num_tiles
        
        vector_dict[f"{x}_mean"] = mean_val
        vector_dict[f"{x}_max"] = max_val
        vector_dict[f"{x}_prevalence"] = prevalence
        
    results.append(vector_dict)

# Создаем датафрейм
df = pd.DataFrame(results)
df = df.round(2) # округляем значения до двух знаков поле запятой

# Порядок колонок (как в пайплайне)
columns_order = ["building_name", "num_tiles"] + \
    [f"{x}_mean" for x in CLASSES_TO_TRACK] + \
    [f"{x}_max" for x in CLASSES_TO_TRACK] + \
    [f"{x}_prevalence" for x in CLASSES_TO_TRACK]

df = df[columns_order]

# Сохраняем в CSV
CSV_OUTPUT = "/kaggle/working/building_features.csv"
df.to_csv(CSV_OUTPUT, index=False)
print(f"\n Извлечено {len(df)} 15D векторов.")
print(f"Сохранено в {CSV_OUTPUT}")
display(df.head())

Агрегация 15D векторов:   0%|          | 0/300 [00:00<?, ?it/s]


 Извлечено 300 15D векторов.
Сохранено в /kaggle/working/building_features.csv


,building_name,num_tiles,coating_deterioration_mean,cracks_mean,masonry_degradation_mean,moisture_bio_damage_mean,vandalism_mean,coating_deterioration_max,cracks_max,masonry_degradation_max,moisture_bio_damage_max,vandalism_max,coating_deterioration_prevalence,cracks_prevalence,masonry_degradation_prevalence,moisture_bio_damage_prevalence,vandalism_prevalence
0,DSC02518,24,0.00,0.0,0.0,0.05,0.0,0.03,0.00,0.01,0.26,0.01,0.08,0.00,0.00,0.38,0.04
1,2026-04-09 17-36-25,12,0.01,0.0,0.0,0.01,0.0,0.03,0.00,0.00,0.04,0.00,0.33,0.00,0.00,0.25,0.00
2,IMG_20260410_142531328_HDR 2,12,0.02,0.0,0.0,0.01,0.0,0.09,0.00,0.01,0.05,0.02,0.50,0.00,0.00,0.25,0.17
3,62,12,0.06,0.0,0.0,0.00,0.0,0.18,0.01,0.02,0.01,0.02,0.67,0.08,0.08,0.00,0.08
4,DSC02333,24,0.00,0.0,0.0,0.01,0.0,0.01,0.00,0.00,0.12,0.00,0.04,0.00,0.00,0.17,0.00
